# Train the fraud detection model

Run the cells from top to bottom (**Runtime → Run all** works too). About 15 minutes in total, mostly training.

1. Loads `train.parquet` from your Drive (saved in Lesson 1), or downloads the data from Kaggle if it's missing.
2. Trains LightGBM with a time-based split and prints the test results.
3. Downloads the trained model as `model.zip`, ready to deploy.

`KAGGLE_API_TOKEN` (key icon in the left sidebar) is only needed if `train.parquet` isn't in your Drive.

## 1 · Install and connect to Drive

In [ ]:
!pip install -q lightgbm==4.6.0
from google.colab import drive, userdata
drive.mount('/content/drive')

from pathlib import Path
PROJECT = Path('/content/drive/MyDrive/fraud-project')
DATA = PROJECT / 'data' / 'train.parquet'
print('train.parquet in Drive:', DATA.exists())

## 2 · Get the data (skipped if Lesson 1 already saved it)

In [ ]:
import os
import pandas as pd

if not DATA.exists():
    os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')
    !pip install -q --upgrade kaggle
    !kaggle competitions download -c ieee-fraud-detection -f train_transaction.csv -p /content/raw
    !kaggle competitions download -c ieee-fraud-detection -f train_identity.csv -p /content/raw
    !cd /content/raw && for f in *.zip; do unzip -oq "$f"; done

    def shrink(df):
        for col in df.columns:
            if df[col].dtype == 'float64':
                df[col] = df[col].astype('float32')
            elif df[col].dtype == 'int64':
                df[col] = pd.to_numeric(df[col], downcast='integer')
        return df

    trans = shrink(pd.read_csv('/content/raw/train_transaction.csv'))
    ident = shrink(pd.read_csv('/content/raw/train_identity.csv'))
    DATA.parent.mkdir(parents=True, exist_ok=True)
    trans.merge(ident, on='TransactionID', how='left').to_parquet(DATA, index=False)
    del trans, ident

print('ready:', DATA, f'{DATA.stat().st_size / 1e6:.0f} MB')

## 3 · Write the project files

In [ ]:
FILES = {
 "fraud/__init__.py": "",
 "fraud/features.py": "\"\"\"Feature preparation shared by training and the API, so both see exactly the same inputs.\"\"\"\n\nimport re\n\nimport numpy as np\nimport pandas as pd\n\n# columns that are never model inputs\nNOT_FEATURES = {\"TransactionID\", \"isFraud\", \"TransactionDT\", \"day\"}\n\n\ndef add_features(df):\n    \"\"\"Add a few simple features on top of the raw columns.\"\"\"\n    out = df.copy()\n    seconds = out[\"TransactionDT\"].astype(\"float64\")\n    out[\"hour\"] = (seconds // 3600) % 24\n    out[\"weekday\"] = (seconds // 86400) % 7\n    amount = out[\"TransactionAmt\"].astype(\"float64\")\n    out[\"amt_cents\"] = (amount - np.floor(amount)).round(3)\n    if \"P_emaildomain\" in out and \"R_emaildomain\" in out:\n        same = out[\"P_emaildomain\"] == out[\"R_emaildomain\"]\n        out[\"same_email_domain\"] = same.where(out[\"R_emaildomain\"].notna()).astype(\"float64\")\n    return out\n\n\ndef to_model_input(df, features, categories):\n    \"\"\"Turn raw rows into the exact table the model was trained on.\"\"\"\n    data = add_features(df).reindex(columns=features)   # missing columns become NaN\n    for col, cats in categories.items():\n        known = data[col].where(data[col].isin(cats))          # values never seen in training become missing\n        data[col] = pd.Categorical(known, categories=cats)\n    for col in features:\n        if col not in categories:\n            data[col] = pd.to_numeric(data[col], errors=\"coerce\").astype(\"float32\")\n    return data\n\n\nID_NAMES = {\"id_30\": \"Operating system\", \"id_31\": \"Browser\", \"id_33\": \"Screen size\",\n            \"id_19\": \"Identity id_19\", \"id_20\": \"Identity id_20\", \"id_13\": \"Identity id_13\"}\n\n\ndef describe(feature, value):\n    \"\"\"Plain-English label for a feature and its value, used to explain a score.\"\"\"\n    shown = \"missing\" if value is None or (isinstance(value, float) and np.isnan(value)) else value\n    if isinstance(shown, float):\n        shown = f\"{shown:g}\"\n    labels = {\n        \"TransactionAmt\": f\"Amount: ${shown}\",\n        \"ProductCD\": f\"Product type: {shown}\",\n        \"card1\": f\"Card group (card1): {shown}\",\n        \"card2\": f\"Card info (card2): {shown}\",\n        \"card3\": f\"Card info (card3): {shown}\",\n        \"card4\": f\"Card network: {shown}\",\n        \"card5\": f\"Card info (card5): {shown}\",\n        \"card6\": f\"Card type: {shown}\",\n        \"addr1\": f\"Billing region: {shown}\",\n        \"addr2\": f\"Billing country: {shown}\",\n        \"P_emaildomain\": f\"Buyer email domain: {shown}\",\n        \"R_emaildomain\": f\"Recipient email domain: {shown}\",\n        \"DeviceType\": f\"Device type: {shown}\",\n        \"DeviceInfo\": f\"Device: {shown}\",\n        \"hour\": f\"Hour of day: {shown}\",\n        \"weekday\": f\"Day of week (0-6): {shown}\",\n        \"amt_cents\": f\"Cents in amount: {shown}\",\n        \"same_email_domain\": f\"Buyer and recipient same email domain: {shown}\",\n    }\n    if feature in labels:\n        return labels[feature]\n    if feature in ID_NAMES:\n        return f\"{ID_NAMES[feature]}: {shown}\"\n    group = re.match(r\"^(C|D|M|V|dist|id_)\\d+$\", feature)\n    kinds = {\"C\": \"Count\", \"D\": \"Days since\", \"M\": \"Match check\", \"V\": \"Vesta feature\",\n             \"dist\": \"Distance\", \"id_\": \"Identity field\"}\n    if group:\n        return f\"{kinds[group.group(1)]} {feature}: {shown}\"\n    return f\"{feature}: {shown}\"\n",
 "fraud/train.py": "\"\"\"Train the fraud model with a time-based split and save everything the API needs.\n\n    python -m fraud.train path/to/train.parquet model/\n\"\"\"\n\nimport json\nimport sys\nimport time\nfrom pathlib import Path\n\nimport lightgbm as lgb\nimport numpy as np\nimport pandas as pd\nfrom sklearn.metrics import average_precision_score, roc_auc_score, roc_curve\n\nfrom fraud.features import NOT_FEATURES, add_features, to_model_input\n\nPARAMS = {\n    \"objective\": \"binary\",\n    \"learning_rate\": 0.08,\n    \"num_leaves\": 128,\n    \"min_child_samples\": 100,\n    \"feature_fraction\": 0.5,\n    \"bagging_fraction\": 0.8,\n    \"bagging_freq\": 1,\n    \"lambda_l2\": 1.0,\n    \"max_cat_to_onehot\": 8,\n    \"cat_smooth\": 20,\n    \"metric\": \"average_precision\",\n    \"verbose\": -1,\n    \"seed\": 42,\n}\n\n\ndef time_split(df, valid_from=0.7, test_from=0.8):\n    \"\"\"Oldest 70% to train, next 10% to tune, newest 20% to test, like a model used on future data.\"\"\"\n    df = df.sort_values(\"TransactionDT\").reset_index(drop=True)\n    a, b = int(len(df) * valid_from), int(len(df) * test_from)\n    return df.iloc[:a], df.iloc[a:b], df.iloc[b:]\n\n\ndef recall_at_fpr(y, scores, threshold):\n    flagged = scores >= threshold\n    fraud, legit = y == 1, y == 0\n    return {\n        \"recall\": float(flagged[fraud].mean()),\n        \"false_positive_rate\": float(flagged[legit].mean()),\n        \"precision\": float(y[flagged].mean()) if flagged.any() else 0.0,\n        \"flagged\": int(flagged.sum()),\n    }\n\n\ndef main(data_path, out_dir, rounds=1500):\n    out = Path(out_dir)\n    out.mkdir(parents=True, exist_ok=True)\n    start = time.time()\n\n    df = pd.read_parquet(data_path)\n    raw_columns = [c for c in df.columns if c not in {\"isFraud\", \"day\"}]\n\n    # text columns (product, card network, email domain, device...) become categories\n    categories = {}\n    for col in raw_columns:\n        if col not in NOT_FEATURES and not pd.api.types.is_numeric_dtype(df[col]):\n            df[col] = df[col].astype(\"object\").where(df[col].isna(), df[col].astype(str))\n            categories[col] = sorted(df[col].dropna().unique().tolist())\n\n    features = [c for c in add_features(df.head(5)).columns if c not in NOT_FEATURES]\n    train, valid, test = time_split(df)\n    print(f\"train {len(train):,}  valid {len(valid):,}  test {len(test):,}  features {len(features)}\")\n\n    X_train = to_model_input(train, features, categories)\n    X_valid = to_model_input(valid, features, categories)\n    X_test = to_model_input(test, features, categories)\n    y_train, y_valid, y_test = (p[\"isFraud\"].to_numpy() for p in (train, valid, test))\n\n    model = lgb.train(\n        PARAMS,\n        lgb.Dataset(X_train, y_train),\n        num_boost_round=rounds,\n        valid_sets=[lgb.Dataset(X_valid, y_valid)],\n        callbacks=[lgb.early_stopping(100), lgb.log_evaluation(100)],\n    )\n\n    p_valid = model.predict(X_valid, num_iteration=model.best_iteration)\n    p_test = model.predict(X_test, num_iteration=model.best_iteration)\n\n    # pick the alert threshold on the validation month, so the test month stays untouched\n    fpr, tpr, thresholds = roc_curve(y_valid, p_valid)\n    threshold = float(thresholds[np.where(fpr <= 0.01)[0][-1]])\n\n    test_fpr, test_tpr, _ = roc_curve(y_test, p_test)\n    metrics = {\n        \"pr_auc\": float(average_precision_score(y_test, p_test)),\n        \"roc_auc\": float(roc_auc_score(y_test, p_test)),\n        \"pr_auc_random_baseline\": float(y_test.mean()),\n        \"recall_at_1pct_fpr\": float(test_tpr[np.where(test_fpr <= 0.01)[0][-1]]),\n        \"at_threshold\": recall_at_fpr(y_test, p_test, threshold),\n        \"threshold\": threshold,\n        \"best_iteration\": int(model.best_iteration),\n        \"rows\": {\"train\": len(train), \"valid\": len(valid), \"test\": len(test)},\n        \"test_fraud_cases\": int(y_test.sum()),\n        \"features\": len(features),\n        \"training_minutes\": round((time.time() - start) / 60, 1),\n    }\n    print(json.dumps(metrics, indent=2))\n\n    model.save_model(out / \"model.txt\", num_iteration=model.best_iteration)\n    meta = {\"features\": features, \"categories\": categories, \"threshold\": threshold, \"raw_columns\": raw_columns}\n    (out / \"meta.json\").write_text(json.dumps(meta))\n    (out / \"metrics.json\").write_text(json.dumps(metrics, indent=2))\n\n    # demo transactions from the test period: a mix of caught, missed and normal ones\n    test = test.assign(score=p_test)\n    fraud = test[test.isFraud == 1]\n    legit = test[test.isFraud == 0]\n    picks = pd.concat([\n        fraud[fraud.score >= threshold].sample(min(30, (fraud.score >= threshold).sum()), random_state=1),\n        fraud[fraud.score < threshold].sample(min(15, (fraud.score < threshold).sum()), random_state=1),\n        legit[legit.score >= threshold].sample(min(10, (legit.score >= threshold).sum()), random_state=1),\n        legit[legit.score < threshold].sample(min(45, (legit.score < threshold).sum()), random_state=1),\n    ]).sample(frac=1, random_state=2)\n    samples = []\n    for _, row in picks.iterrows():\n        values = {c: (v.item() if hasattr(v, \"item\") else v) for c, v in row[raw_columns].items() if pd.notna(v)}\n        samples.append({\"id\": int(row[\"TransactionID\"]), \"is_fraud\": int(row[\"isFraud\"]), \"values\": values})\n    (out / \"samples.json\").write_text(json.dumps(samples))\n    print(f\"saved model, meta, metrics and {len(samples)} samples to {out}\")\n    return metrics\n\n\nif __name__ == \"__main__\":\n    main(sys.argv[1], sys.argv[2])\n",
 "app/__init__.py": "",
 "app/main.py": "\"\"\"FastAPI service: scores a card transaction and explains the score.\"\"\"\n\nimport gzip\nimport json\nimport math\nimport os\nfrom pathlib import Path\n\nimport lightgbm as lgb\nimport numpy as np\nimport pandas as pd\nfrom fastapi import FastAPI, HTTPException\nfrom fastapi.responses import FileResponse\nfrom fastapi.staticfiles import StaticFiles\nfrom pydantic import BaseModel\n\nfrom fraud.features import describe, to_model_input\n\nROOT = Path(__file__).resolve().parent.parent\nMODEL_DIR = Path(os.environ.get(\"MODEL_DIR\", ROOT / \"model\"))\n\n\ndef load_model():\n    \"\"\"The model is stored gzipped to stay small in git; a plain model.txt works too.\"\"\"\n    packed = MODEL_DIR / \"model.txt.gz\"\n    if packed.exists():\n        return lgb.Booster(model_str=gzip.decompress(packed.read_bytes()).decode())\n    return lgb.Booster(model_file=str(MODEL_DIR / \"model.txt\"))\n\n\nmodel = load_model()\nmeta = json.loads((MODEL_DIR / \"meta.json\").read_text())\nmetrics = json.loads((MODEL_DIR / \"metrics.json\").read_text())\nsamples = json.loads((MODEL_DIR / \"samples.json\").read_text())\nsamples_by_id = {s[\"id\"]: s for s in samples}\nFEATURES, CATEGORIES, THRESHOLD = meta[\"features\"], meta[\"categories\"], meta[\"threshold\"]\n\napp = FastAPI(title=\"Fraud Detection API\", version=\"1.0\")\n\n\nclass Transaction(BaseModel):\n    values: dict\n\n\ndef clean(value):\n    if value is None or (isinstance(value, float) and math.isnan(value)):\n        return None\n    return value.item() if hasattr(value, \"item\") else value\n\n\ndef score(values):\n    \"\"\"Probability of fraud, the decision at the alert threshold, and the top reasons.\"\"\"\n    for field in (\"TransactionAmt\", \"TransactionDT\"):\n        try:\n            number = float(values[field])\n        except KeyError:\n            raise HTTPException(422, f\"values must include {field}\")\n        except (TypeError, ValueError):\n            raise HTTPException(422, f\"{field} must be a number\")\n        if not math.isfinite(number) or number < 0 or (field == \"TransactionAmt\" and number == 0):\n            raise HTTPException(422, f\"{field} must be a positive number\")\n    row = pd.DataFrame([values])\n    for col in CATEGORIES:\n        if col in row:\n            row[col] = row[col].map(lambda v: None if v in (None, \"\") else str(v))\n    X = to_model_input(row, FEATURES, CATEGORIES)\n\n    probability = float(model.predict(X)[0])\n    # SHAP values from LightGBM itself: one contribution per feature, plus a baseline at the end\n    contributions = model.predict(X, pred_contrib=True)[0][:-1]\n    order = np.argsort(-np.abs(contributions))[:6]\n    raw = X.iloc[0]\n    reasons = [{\n        \"feature\": FEATURES[i],\n        \"label\": describe(FEATURES[i], clean(raw.iloc[i])),\n        \"impact\": round(float(contributions[i]), 4),\n    } for i in order if contributions[i] != 0]\n\n    return {\n        \"fraud_probability\": round(probability, 4),\n        \"flagged\": probability >= THRESHOLD,\n        \"threshold\": round(THRESHOLD, 4),\n        \"reasons\": reasons,\n    }\n\n\n@app.get(\"/api/health\")\ndef health():\n    return {\"status\": \"ok\", \"features\": len(FEATURES)}\n\n\n@app.get(\"/api/metrics\")\ndef get_metrics():\n    return metrics\n\n\n@app.get(\"/api/samples\")\ndef list_samples():\n    return [{\n        \"id\": s[\"id\"],\n        \"is_fraud\": s[\"is_fraud\"],\n        \"amount\": s[\"values\"].get(\"TransactionAmt\"),\n        \"product\": s[\"values\"].get(\"ProductCD\"),\n        \"card\": s[\"values\"].get(\"card4\"),\n    } for s in samples]\n\n\n@app.get(\"/api/samples/{sample_id}\")\ndef get_sample(sample_id: int):\n    if sample_id not in samples_by_id:\n        raise HTTPException(404, \"no such sample\")\n    return samples_by_id[sample_id]\n\n\n@app.post(\"/api/score\")\ndef score_transaction(transaction: Transaction):\n    return score(transaction.values)\n\n\napp.mount(\"/static\", StaticFiles(directory=ROOT / \"app\" / \"static\"), name=\"static\")\n\n\n@app.get(\"/\")\ndef home():\n    return FileResponse(ROOT / \"app\" / \"static\" / \"index.html\")\n",
 "app/static/index.html": "<!doctype html>\n<html lang=\"en\">\n<head>\n<meta charset=\"utf-8\">\n<meta name=\"viewport\" content=\"width=device-width, initial-scale=1\">\n<title>Card Fraud Detection</title>\n<link rel=\"preconnect\" href=\"https://fonts.googleapis.com\">\n<link href=\"https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700&family=JetBrains+Mono:wght@500&display=swap\" rel=\"stylesheet\">\n<style>\n  :root {\n    --bg: #f6f7f9; --panel: #ffffff; --ink: #14171f; --muted: #5d6475; --line: #e3e6ec;\n    --accent: #2f5bea; --danger: #d4372c; --danger-soft: #fdecea; --safe: #1f8a4c; --safe-soft: #e7f5ed;\n  }\n  @media (prefers-color-scheme: dark) {\n    :root { --bg: #0f1218; --panel: #171b23; --ink: #e8eaf0; --muted: #9aa2b4; --line: #272c37;\n            --accent: #6d8dff; --danger: #ff6b5f; --danger-soft: #3a1d1b; --safe: #4cc985; --safe-soft: #16301f; }\n  }\n  * { box-sizing: border-box; }\n  body { margin: 0; background: var(--bg); color: var(--ink); font: 15px/1.5 Inter, system-ui, sans-serif; }\n  .wrap { max-width: 1120px; margin: 0 auto; padding: 32px 16px 48px; }\n  header h1 { margin: 0; font-size: 28px; letter-spacing: -0.02em; }\n  header p { margin: 6px 0 0; color: var(--muted); max-width: 720px; }\n  .num { font-family: \"JetBrains Mono\", ui-monospace, monospace; }\n  .metrics { display: grid; grid-template-columns: repeat(4, 1fr); gap: 12px; margin: 24px 0; }\n  .metric { background: var(--panel); border: 1px solid var(--line); border-radius: 12px; padding: 14px 16px; }\n  .metric .value { font-size: 26px; font-weight: 700; }\n  .metric .label { font-size: 13px; color: var(--muted); }\n  .grid { display: grid; grid-template-columns: 340px 1fr; gap: 16px; align-items: start; }\n  .panel { background: var(--panel); border: 1px solid var(--line); border-radius: 12px; padding: 16px; }\n  .panel h2 { margin: 0 0 12px; font-size: 15px; }\n  .filters { display: flex; gap: 6px; margin-bottom: 10px; }\n  .filters button { flex: 1; }\n  button { font: inherit; border: 1px solid var(--line); background: var(--panel); color: var(--ink);\n           border-radius: 8px; padding: 6px 10px; cursor: pointer; }\n  button.on { border-color: var(--accent); color: var(--accent); font-weight: 600; }\n  button.primary { background: var(--accent); border-color: var(--accent); color: #fff; font-weight: 600; padding: 9px 16px; }\n  .list { max-height: 560px; overflow-y: auto; margin: 0 -6px; }\n  .item { display: grid; grid-template-columns: 1fr auto; gap: 2px 8px; padding: 9px 10px; margin: 0 6px 4px;\n          border-radius: 8px; cursor: pointer; border: 1px solid transparent; }\n  .item:hover { background: var(--bg); }\n  .item.sel { border-color: var(--accent); background: var(--bg); }\n  .item .sub { font-size: 12px; color: var(--muted); grid-column: 1 / -1; }\n  .tag { font-size: 12px; font-weight: 600; padding: 1px 8px; border-radius: 99px; align-self: center; }\n  .tag.fraud { color: var(--danger); background: var(--danger-soft); }\n  .tag.legit { color: var(--safe); background: var(--safe-soft); }\n  .fields { display: grid; grid-template-columns: repeat(3, 1fr); gap: 12px; }\n  label { font-size: 12px; color: var(--muted); display: block; }\n  input, select { width: 100%; margin-top: 4px; font: inherit; padding: 7px 9px; border-radius: 8px;\n                  border: 1px solid var(--line); background: var(--bg); color: var(--ink); }\n  .actions { display: flex; align-items: center; gap: 12px; margin-top: 14px; flex-wrap: wrap; }\n  .hint { font-size: 13px; color: var(--muted); }\n  .hint.error { color: var(--danger); font-weight: 500; }\n  .result { margin-top: 16px; border-top: 1px solid var(--line); padding-top: 16px; display: none; }\n  .verdict { display: flex; align-items: baseline; gap: 14px; flex-wrap: wrap; }\n  .prob { font-size: 40px; font-weight: 700; }\n  .pill { font-weight: 600; padding: 4px 12px; border-radius: 99px; }\n  .pill.flag { color: var(--danger); background: var(--danger-soft); }\n  .pill.ok { color: var(--safe); background: var(--safe-soft); }\n  .bar-track { height: 8px; background: var(--bg); border-radius: 99px; margin: 10px 0 4px; position: relative; }\n  .bar-fill { height: 100%; border-radius: 99px; background: var(--accent); }\n  .bar-mark { position: absolute; top: -4px; width: 2px; height: 16px; background: var(--ink); }\n  .reasons { margin-top: 16px; }\n  .reason { display: grid; grid-template-columns: 1fr 160px; gap: 10px; align-items: center; padding: 5px 0; font-size: 14px; }\n  .rbar { height: 10px; border-radius: 3px; }\n  .rbar.up { background: var(--danger); } .rbar.down { background: var(--safe); }\n  .how { margin-top: 24px; color: var(--muted); font-size: 14px; }\n  .how b { color: var(--ink); }\n  a { color: var(--accent); }\n  @media (max-width: 820px) {\n    .metrics { grid-template-columns: repeat(2, 1fr); }\n    .grid { grid-template-columns: 1fr; }\n    .fields { grid-template-columns: 1fr 1fr; }\n    .list { max-height: 300px; }\n  }\n</style>\n</head>\n<body>\n<div class=\"wrap\">\n  <header>\n    <h1>Card Fraud Detection</h1>\n    <p>A LightGBM model trained on the IEEE-CIS dataset of real e-commerce card transactions. It was trained on the\n       oldest transactions and tested only on the newest ones, the way a fraud model is used in practice.</p>\n  </header>\n\n  <section class=\"metrics\" id=\"metrics\"></section>\n\n  <section class=\"grid\">\n    <div class=\"panel\">\n      <h2>Transactions from the test period</h2>\n      <div class=\"filters\">\n        <button data-f=\"all\" class=\"on\">All</button>\n        <button data-f=\"1\">Fraud</button>\n        <button data-f=\"0\">Legit</button>\n      </div>\n      <div class=\"list\" id=\"list\"></div>\n    </div>\n\n    <div class=\"panel\">\n      <h2 id=\"title\">Pick a transaction</h2>\n      <div class=\"fields\" id=\"fields\"></div>\n      <div class=\"actions\">\n        <button class=\"primary\" id=\"score\">Score transaction</button>\n        <span class=\"hint\" id=\"hint\">Change any field and score it again to see how the model reacts.</span>\n      </div>\n      <div class=\"result\" id=\"result\">\n        <div class=\"verdict\">\n          <span class=\"prob num\" id=\"prob\"></span>\n          <span class=\"pill\" id=\"pill\"></span>\n          <span class=\"hint\" id=\"actual\"></span>\n        </div>\n        <div class=\"bar-track\"><div class=\"bar-fill\" id=\"fill\"></div><div class=\"bar-mark\" id=\"mark\"></div></div>\n        <div class=\"hint\" id=\"threshold-note\"></div>\n        <div class=\"reasons\">\n          <h2>Why: biggest factors in this score (SHAP)</h2>\n          <div id=\"reasons\"></div>\n        </div>\n      </div>\n    </div>\n  </section>\n\n  <section class=\"how\">\n    <p><b>How it works.</b> Each transaction has about 430 columns: amount, product, card details, billing region,\n       email domains, device, plus count, time-gap and match features from Vesta, the company that released the data.\n       The model returns a fraud probability. A transaction is flagged when that probability passes a threshold chosen\n       so that only 1% of legitimate transactions get flagged, set on a validation month the test never saw.\n       The factors come from SHAP values: red pushed the score up, green pulled it down.</p>\n    <p><a href=\"/docs\">API docs</a> &middot; <a id=\"repo\" href=\"https://github.com/dishasharma23-prog/fraud-detection\">Source on GitHub</a></p>\n  </section>\n</div>\n<script src=\"/static/app.js\"></script>\n</body>\n</html>\n",
 "app/static/app.js": "const FIELDS = [\n  [\"TransactionAmt\", \"Amount (USD)\", \"number\"],\n  [\"ProductCD\", \"Product type\", [\"W\", \"C\", \"H\", \"R\", \"S\"]],\n  [\"card4\", \"Card network\", [\"visa\", \"mastercard\", \"american express\", \"discover\"]],\n  [\"card6\", \"Card type\", [\"debit\", \"credit\"]],\n  [\"P_emaildomain\", \"Buyer email domain\", \"text\"],\n  [\"DeviceType\", \"Device type\", [\"desktop\", \"mobile\"]],\n];\n\nlet samples = [];\nlet current = null;\nlet filter = \"all\";\n\nconst $ = (id) => document.getElementById(id);\nconst pct = (x) => (x * 100).toFixed(1) + \"%\";\n\nconst HINT = \"Change any field and score it again to see how the model reacts.\";\n\nasync function getJSON(url, options) {\n  const res = await fetch(url, options);\n  if (!res.ok) {\n    let message = `Request failed (${res.status})`;\n    try { message = (await res.json()).detail ?? message; } catch (e) {}\n    throw new Error(typeof message === \"string\" ? message : \"Some fields are not valid\");\n  }\n  return res.json();\n}\n\nfunction showHint(text, isError) {\n  $(\"hint\").textContent = text;\n  $(\"hint\").classList.toggle(\"error\", Boolean(isError));\n}\n\nfunction showMetrics(m) {\n  const cards = [\n    [m.pr_auc.toFixed(3), `PR-AUC (random guessing: ${m.pr_auc_random_baseline.toFixed(3)})`],\n    [m.roc_auc.toFixed(3), \"ROC-AUC\"],\n    [pct(m.at_threshold.recall), `of fraud caught, with ${pct(m.at_threshold.false_positive_rate)} of legit flagged`],\n    [m.rows.test.toLocaleString(), `test transactions (${m.test_fraud_cases.toLocaleString()} fraud)`],\n  ];\n  $(\"metrics\").innerHTML = cards.map(([v, l]) =>\n    `<div class=\"metric\"><div class=\"value num\">${v}</div><div class=\"label\">${l}</div></div>`).join(\"\");\n}\n\nfunction showList() {\n  const rows = samples.filter((s) => filter === \"all\" || String(s.is_fraud) === filter);\n  $(\"list\").innerHTML = rows.map((s) => `\n    <div class=\"item ${current && current.id === s.id ? \"sel\" : \"\"}\" data-id=\"${s.id}\">\n      <span class=\"num\">#${s.id}</span>\n      <span class=\"tag ${s.is_fraud ? \"fraud\" : \"legit\"}\">${s.is_fraud ? \"fraud\" : \"legit\"}</span>\n      <span class=\"sub\">$${Number(s.amount).toFixed(2)} · product ${s.product ?? \"?\"} · ${s.card ?? \"unknown card\"}</span>\n    </div>`).join(\"\");\n}\n\nfunction field(key, label, kind, value) {\n  const v = value ?? \"\";\n  if (Array.isArray(kind)) {\n    const opts = [\"\", ...kind].map((o) =>\n      `<option value=\"${o}\" ${String(o) === String(v) ? \"selected\" : \"\"}>${o || \"missing\"}</option>`).join(\"\");\n    return `<div><label>${label}<select data-key=\"${key}\">${opts}</select></label></div>`;\n  }\n  const shown = kind === \"number\" && v !== \"\" ? Number(v).toFixed(2) : v;\n  return `<div><label>${label}<input data-key=\"${key}\" type=\"${kind}\" step=\"0.01\" value=\"${shown}\"></label></div>`;\n}\n\nasync function pick(id) {\n  current = await getJSON(`/api/samples/${id}`);\n  $(\"title\").textContent = `Transaction #${current.id}`;\n  $(\"fields\").innerHTML = FIELDS.map(([k, l, kind]) => field(k, l, kind, current.values[k])).join(\"\");\n  showList();\n  await scoreCurrent();\n}\n\nfunction edited() {\n  const values = { ...current.values };\n  let changed = false;\n  document.querySelectorAll(\"[data-key]\").forEach((el) => {\n    const key = el.dataset.key;\n    let v = el.value === \"\" ? null : el.value;\n    if (el.type === \"number\" && v !== null) v = Number(v);\n    const before = current.values[key] ?? null;\n    const same = el.type === \"number\" ? Math.abs((before ?? 0) - (v ?? 0)) < 0.005 && (before === null) === (v === null) : before === v;\n    if (!same) changed = true;\n    if (v === null) delete values[key]; else values[key] = v;\n  });\n  return { values, changed };\n}\n\nasync function scoreCurrent() {\n  if (!current) return;\n  const { values, changed } = edited();\n  let r;\n  try {\n    r = await getJSON(\"/api/score\", {\n      method: \"POST\",\n      headers: { \"Content-Type\": \"application/json\" },\n      body: JSON.stringify({ values }),\n    });\n  } catch (err) {\n    $(\"result\").style.display = \"none\";\n    showHint(`Can't score this: ${err.message}.`, true);\n    return;\n  }\n  showHint(HINT, false);\n  $(\"result\").style.display = \"block\";\n  $(\"prob\").textContent = pct(r.fraud_probability);\n  $(\"pill\").textContent = r.flagged ? \"Flag for review\" : \"Approve\";\n  $(\"pill\").className = \"pill \" + (r.flagged ? \"flag\" : \"ok\");\n  $(\"actual\").textContent = changed ? \"Edited transaction\" : `Actually: ${current.is_fraud ? \"fraud\" : \"legit\"}`;\n  $(\"fill\").style.width = Math.max(1, r.fraud_probability * 100) + \"%\";\n  $(\"fill\").style.background = r.flagged ? \"var(--danger)\" : \"var(--safe)\";\n  $(\"mark\").style.left = r.threshold * 100 + \"%\";\n  $(\"threshold-note\").textContent = `Flagged above ${pct(r.threshold)} (the black line)`;\n  const biggest = Math.max(...r.reasons.map((x) => Math.abs(x.impact)), 1e-9);\n  $(\"reasons\").innerHTML = r.reasons.map((x) => `\n    <div class=\"reason\">\n      <span>${x.label}</span>\n      <div class=\"rbar ${x.impact > 0 ? \"up\" : \"down\"}\" style=\"width:${(Math.abs(x.impact) / biggest) * 100}%\"></div>\n    </div>`).join(\"\");\n}\n\ndocument.querySelectorAll(\".filters button\").forEach((b) => b.addEventListener(\"click\", () => {\n  filter = b.dataset.f;\n  document.querySelectorAll(\".filters button\").forEach((x) => x.classList.toggle(\"on\", x === b));\n  showList();\n}));\n$(\"list\").addEventListener(\"click\", (e) => {\n  const item = e.target.closest(\".item\");\n  if (item) pick(Number(item.dataset.id));\n});\n$(\"score\").addEventListener(\"click\", scoreCurrent);\n\n(async () => {\n  showMetrics(await getJSON(\"/api/metrics\"));\n  samples = await getJSON(\"/api/samples\");\n  showList();\n  if (samples.length) pick(samples.find((s) => s.is_fraud)?.id ?? samples[0].id);\n})();\n",
 "requirements.txt": "fastapi==0.115.6\nuvicorn==0.34.0\nlightgbm==4.6.0\npandas>=2.0,<3\nnumpy>=1.24,<3\n",
 "Dockerfile": "FROM python:3.11-slim\n\n# LightGBM needs the OpenMP runtime\nRUN apt-get update && apt-get install -y --no-install-recommends libgomp1 && rm -rf /var/lib/apt/lists/*\n\nRUN useradd -m -u 1000 user\nUSER user\nWORKDIR /home/user/app\n\nCOPY --chown=user requirements.txt .\nRUN pip install --no-cache-dir -r requirements.txt\n\nCOPY --chown=user . .\n\n# hosts like Render pass the port in $PORT; 7860 when run locally\nENV PORT=7860\nEXPOSE 7860\nCMD uvicorn app.main:app --host 0.0.0.0 --port $PORT\n",
 ".dockerignore": "__pycache__\n*.pyc\n.git\nnotebooks\ndata\n",
 "README.md": "# Card Fraud Detection\n\nA LightGBM model that scores card transactions for fraud, served by a FastAPI service with a small web page.\nEvery score comes with the factors that drove it (SHAP values).\n\n**Live demo:** https://dishasharma23-prog.github.io/fraud-detection/ (the model's scores and SHAP explanations for 100 transactions from the test period)\n\nTrained on the [IEEE-CIS Fraud Detection](https://www.kaggle.com/c/ieee-fraud-detection) dataset:\n590,540 real e-commerce transactions from Vesta, 3.5% of them fraud.\n\n## How it was built\n\n1. **Time-based split.** Transactions are sorted by time: the oldest 70% train the model, the next 10% are used to stop\n   training and pick the alert threshold, and the newest 20% are kept untouched for the final test. A random split would\n   let the model peek at the future and inflate the scores.\n2. **Model.** LightGBM (gradient-boosted trees) on all ~430 columns, with text columns such as product, card network,\n   email domain and device treated as categories, plus four extra features: hour of day, day of week, the cents in the\n   amount, and whether buyer and recipient share an email domain.\n3. **Metrics that fit rare fraud.** With 3.5% fraud, a model that always says \"legit\" is 96.5% accurate, so accuracy is\n   not reported. The results use PR-AUC, ROC-AUC, and recall when only 1% of legitimate transactions may be flagged.\n4. **Threshold.** A transaction is flagged when its probability passes the threshold that flags 1% of legitimate\n   transactions in the validation period.\n5. **Explanations.** LightGBM computes SHAP values directly (`pred_contrib=True`), so the API returns the top factors\n   for every score without a separate SHAP model.\n\nResults on the test period are in `model/metrics.json` and on the live page.\n\n## API\n\n| Method | Path | What it does |\n|---|---|---|\n| GET | `/api/health` | service check |\n| GET | `/api/metrics` | test-set metrics |\n| GET | `/api/samples` | demo transactions from the test period |\n| GET | `/api/samples/{id}` | one demo transaction with all its values |\n| POST | `/api/score` | `{\"values\": {...}}` → fraud probability, flag, top reasons |\n\nInteractive docs at `/docs`.\n\n## Tests\n\n```bash\npip install -r requirements-dev.txt\npython -m playwright install chromium\npython -m pytest -v\n```\n\n20 tests run on every push (GitHub Actions):\n\n- **API (14):** normal scoring, missing or invalid fields (text, negative or empty amounts get a 422), product or card\n  values the model has never seen, empty strings treated as missing, unknown IDs, and a regression check that the API\n  returns the exact scores published on the demo page.\n- **Browser (6, Playwright):** the demo page shows the real metrics, lists and filters every transaction, and shows the\n  right score, decision and reasons for each one; it fits a phone screen; the API page re-scores an edited transaction\n  and explains invalid input instead of failing silently.\n\n## Run it locally\n\n```bash\npip install -r requirements.txt\nuvicorn app.main:app --port 7860\n```\n\nThe model files in `model/` (`model.txt.gz`, `meta.json`, `metrics.json`, `samples.json`) come from\n`notebooks/train_and_deploy.ipynb`, which trains on the Kaggle data in Google Colab.\n\nOr with Docker:\n\n```bash\ndocker build -t fraud-detection .\ndocker run -p 7860:7860 fraud-detection\n```\n\n## Project layout\n\n```\nfraud/features.py   feature preparation shared by training and the API\nfraud/train.py      time split, LightGBM training, metrics, demo samples\napp/main.py         FastAPI service\napp/static/         web page served by the API\ntests/              API and browser tests (pytest, Playwright)\nsite/build_site.py  builds the static demo in docs/ (GitHub Pages) from the trained model\nnotebooks/          Colab notebook that trains the model\nmodel/              trained model and its test results\nDockerfile          container for the API\n```\n\nBuilt by Disha Sharma.\n"
}

APP = Path('/content/fraud-detection')
for name, text in FILES.items():
    path = APP / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")
print('wrote', len(FILES), 'files to', APP)

## 4 · Train and test

Oldest 70% of transactions train the model, the next 10% stop training and set the alert threshold, and the newest 20%
are the test. Takes roughly 5 to 15 minutes. You'll see the validation PR-AUC printed every 100 rounds.

In [ ]:
import sys, shutil
sys.path.insert(0, str(APP))
from fraud.train import main

metrics = main(DATA, APP / 'model')
shutil.copytree(APP / 'model', PROJECT / 'model', dirs_exist_ok=True)   # keep a copy in Drive

## 5 · Results in plain words (copy these into your resume and README)

In [ ]:
m, t = metrics, metrics['at_threshold']
print(f"Test set: newest {m['rows']['test']:,} transactions, {m['test_fraud_cases']:,} of them fraud")
print(f"PR-AUC {m['pr_auc']:.3f} (random guessing scores {m['pr_auc_random_baseline']:.3f})")
print(f"ROC-AUC {m['roc_auc']:.3f}")
print(f"At the alert threshold: catches {t['recall']:.1%} of fraud while flagging {t['false_positive_rate']:.2%} of legit transactions")
print(f"Of the transactions it flags, {t['precision']:.1%} are actually fraud")

## 6 · Download the model

Compresses the model and downloads `model.zip` (model, test results and demo transactions).
Its contents go in the `model/` folder of the GitHub repo, which Render deploys.

In [ ]:
import gzip, shutil
from google.colab import files

model_dir = APP / 'model'
with open(model_dir / 'model.txt', 'rb') as src, gzip.open(model_dir / 'model.txt.gz', 'wb') as dst:
    shutil.copyfileobj(src, dst)
bundle = Path('/content/model_files')
shutil.rmtree(bundle, ignore_errors=True)
bundle.mkdir()
for name in ['model.txt.gz', 'meta.json', 'metrics.json', 'samples.json']:
    shutil.copy(model_dir / name, bundle / name)
    print(f"{name:14} {(bundle / name).stat().st_size / 1e6:6.1f} MB")
shutil.make_archive('/content/model', 'zip', bundle)
files.download('/content/model.zip')